# Breast signature analysis -- run + view (inline, parameterised)

Self-contained: load the breast adata, add the cancer gene-set signatures as
spatial-neighborhood factors, then for each target signature **fit + resample + plot** live
in the notebook (no SLURM cache). Change the settings cell to re-run with different
parameters (e.g. `METHOD='ridge', PENALTY=800`). This is the old `signature_analysis`-style
workflow; the cache-backed `signature_analysis_breast.ipynb` viewer is separate.

## Settings

In [1]:
import sys
from pathlib import Path
import json

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import scanpy as sc
from metab_processing.LinearRegression import signature_helpers as sh
from metab_processing.LinearRegression.signature_configs import CONFIGS
from metab_processing.LinearRegression.build_x import ensure_lognorm_layer

cfg = CONFIGS['breast']

# ---- run parameters (edit + re-run to try different fits) ----
METHOD, PENALTY, STANDARDIZE = 'ridge', 100, True     # e.g. 'ridge', 800, True
N_SUB = 200
LOWER, UPPER = 15, 85
TOP = 40
PLOTS = ('ratio', 'variance_ratio', 'meansq_variance_ratio')   # which ratio plots to draw

## Load the breast data

In [2]:
adata = sc.read_h5ad(cfg.adata_path)
ensure_lognorm_layer(adata)
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns.get('x_genes', [])), 'built genes')
print('cell types:', sorted(adata.obs[cfg.cell_type_col].astype(str).unique()))

170045 cells | 18028 built genes
cell types: ['11q13 High Grade DCIS Cells', '11q13 High Grade DCIS Cells (Mitotic)', '11q13 High Grade DCIS Tumor Cells (G1/S)', 'Apocrine Cells', 'Basal-like Structured DCIS Cells', 'CAFs, High Grade DCIS Associated', 'CAFs, Low Grade DCIS Associated', 'CXCL14+ Fibroblasts', 'Dendritic Cells', 'Endothelial Cells', 'Luminal-like Amorphous DCIS Cells', 'Macrophages', 'Mast Cells', 'Myeloid Cells', 'Myoepithelial Cells', 'Pericytes', 'Plasma & Mast Cell Mixture', 'Plasma Cells', 'T Lymphocytes']


## Gene sets: targets (predicted) and cancer factors (added as neighborhood predictors)

In [3]:
library = sh.load_msigdb(cfg.dbver, cfg.categories)

# Target signatures (predicted pseudo-genes): MSigDB targets + manual literal sets.
TARGET_NAMES = list(cfg.target_signatures) + list(cfg.neg_controls)
sets = sh.resolve_sets(TARGET_NAMES, library, var, label='breast targets')
for name, genes in cfg.manual_sets.items():                 # manual (non-MSigDB) sets
    sets[name] = [g for g in genes if g in var]
    print(f'{name}: {len(sets[name])}/{len(genes)} present (manual)')

sh.report_factor_removal(adata, sets)

TARGETS = TARGET_NAMES + list(cfg.manual_sets)

===== breast targets: membership vs data (17 sets) =====

HALLMARK_IL2_STAT5_SIGNALING  (199/199 present)
  in signature (199): ['ABCB1', 'ADAM19', 'AGER', 'AHCY', 'AHNAK', 'AHR', 'ALCAM', 'AMACR', 'ANXA4', 'APLP1', 'ARL4A', 'BATF', 'BATF3', 'BCL2', 'BCL2L1', 'BHLHE40', 'BMP2', 'BMPR2', 'CA2', 'CAPG', 'CAPN3', 'CASP3', 'CCND2', 'CCND3', 'CCNE1', 'CCR4', 'CD44', 'CD48', 'CD79B', 'CD81', 'CD83', 'CD86', 'CDC42SE2', 'CDC6', 'CDCP1', 'CDKN1C', 'CISH', 'CKAP4', 'COCH', 'COL6A1', 'CSF1', 'CSF2', 'CST7', 'CTLA4', 'CTSZ', 'CXCL10', 'CYFIP1', 'DCPS', 'DENND5A', 'DHRS3', 'DRC1', 'ECM1', 'EEF1AKMT1', 'EMP1', 'ENO3', 'ENPP1', 'EOMES', 'ETFBKMT', 'ETV4', 'F2RL2', 'FAH', 'FGL2', 'FLT3LG', 'FURIN', 'GABARAPL1', 'GADD45B', 'GALM', 'GATA1', 'GBP4', 'GLIPR2', 'GPR65', 'GPR83', 'GPX4', 'GSTO1', 'GUCY1B1', 'HIPK2', 'HK2', 'HOPX', 'HUWE1', 'HYCC2', 'ICOS', 'IFITM3', 'IFNGR1', 'IGF1R', 'IGF2R', 'IKZF2', 'IKZF4', 'IL10', 'IL10RA', 'IL13', 'IL18R1', 'IL1R2', 'IL1RL1', 'IL2RA', 'IL2RB', 'IL3RA', 'IL4R', 'IRF4'

In [4]:
# Cancer-associated gene sets -> spatial-neighborhood FACTORS (split by cell type) + proximity.
cancer_sets = {n: [g for g in library.get(n, []) if g in var] for n in cfg.cancer_factors}
cancer_sets.update({v: [g for g in gl if g in var] for v, gl in cfg.vessel.items()})
cancer_sets = {k: v for k, v in cancer_sets.items() if v}
sh.add_cancer_factors(adata, cancer_sets, annot_col=cfg.cell_type_col, add_proximity=True)

add_cancer_factors: added/updated custom factors; x_custom now has 418 columns from 21 signature(s).


AnnData object with n_obs × n_vars = 170045 × 18028
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'graphclust', 'cell_type'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'cell_type_colors', 'x_factor_map', 'x_factors_cols', 'x_genes', 'x_custom_cols'
    obsm: 'X_pca', 'X_umap', 'spatial', 'x_factors', 'x_custom'
    layers: 'normalized_count', 'raw_count'

## Run a signature

`run(sig)` = add the pseudo-gene -> fit (METHOD/PENALTY) on the breast T cells -> resample -> ranked table + the selected ratio plots. `custom='all'` includes the cancer neighborhood factors.

In [5]:
def run(sig):
    return sh.run_signature(adata, sig, sets, cell_types=list(cfg.t_cell_types),
                            cell_type_col=cfg.cell_type_col, method=METHOD, penalty=PENALTY,
                            standardize=STANDARDIZE, n_sub=N_SUB, lower=LOWER, upper=UPPER,
                            top=TOP, plots=PLOTS)

## T-cell program targets

In [ ]:
run('HALLMARK_IL2_STAT5_SIGNALING')

[add_gene_signature 'HALLMARK_IL2_STAT5_SIGNALING'] factor_mode='union' keep_when_overlap='factor'
  dropped 67 gene(s) from the target: ['AHR', 'ALCAM', 'BATF', 'BATF3', 'BHLHE40', 'BMP2', 'BMPR2', 'CCR4', 'CD44', 'CD48', 'CD86', 'COL6A1', 'CSF1', 'CSF2', 'CTLA4', 'CXCL10', 'EOMES', 'ETV4', 'F2RL2', 'FLT3LG', 'GATA1', 'GPR83', 'ICOS', 'IFNGR1', 'IGF1R', 'IGF2R', 'IKZF2', 'IKZF4', 'IL10', 'IL10RA', 'IL13', 'IL18R1', 'IL1R2', 'IL1RL1', 'IL2RB', 'IL3RA', 'IL4R', 'IRF4', 'IRF6', 'IRF8', 'ITGA6', 'ITGAE', 'ITGAV', 'KLF6', 'LIF', 'LTB', 'MAFF', 'MYC', 'NFIL3', 'NRP1', 'PENK', 'PLAGL1', 'POU2F1', 'PTCH1', 'PTH1R', 'RORA', 'SELL', 'SELP', 'SPP1', 'TNFRSF18', 'TNFRSF1B', 'TNFRSF4', 'TNFRSF8', 'TNFRSF9', 'TNFSF10', 'TNFSF11', 'XBP1']
  kept all 6553 factor(s)

===== HALLMARK_IL2_STAT5_SIGNALING | T Lymphocytes only =====


In [ ]:
run('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')

In [ ]:
run('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')

In [ ]:
run('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')

In [ ]:
run('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

In [ ]:
run('HALLMARK_INTERFERON_GAMMA_RESPONSE')

In [ ]:
run('HALLMARK_INTERFERON_ALPHA_RESPONSE')

In [ ]:
run('GOBP_CELL_KILLING')

In [ ]:
run('GOBP_GRANZYME_MEDIATED_PROGRAMMED_CELL_DEATH_SIGNALING_PATHWAY')

In [ ]:
run('GOBP_LYMPHOCYTE_MEDIATED_IMMUNITY')

## Metabolic targets

In [ ]:
run('HALLMARK_HYPOXIA')

In [ ]:
run('HALLMARK_GLYCOLYSIS')

In [ ]:
run('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
run('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
run('HALLMARK_MTORC1_SIGNALING')

## Negative controls

In [ ]:
run('HALLMARK_PANCREAS_BETA_CELLS')

In [ ]:
run('HALLMARK_SPERMATOGENESIS')

In [ ]:
run('MANUAL_ISLET_NEG_CONTROL')